# Part 2: Create Regattas in RailMeets

In [5]:
import pandas as pd
from datetime import datetime
import os

In [16]:
year = datetime.now().year

morf = pd.read_csv(f"morf_schedule/morf_schedule_{year}.csv")

railmeets = pd.DataFrame()

railmeets["Event Date"] = pd.to_datetime(morf["Date"]).dt.date
railmeets["Event Time"] = morf["Start Time"]

railmeets["Timezone"] = "America/Chicago"
railmeets["Fleet"] = "MORF"
railmeets["Sponsoring Club"] = "Midwest Open Racing Fleet"

railmeets["Series"] = "MORF " + morf["Series Type"].fillna("")
railmeets["Event"] = morf["Event - Results"]
railmeets["Race Type"] = morf["Race Type"]

railmeets["Location"] = "Chicago IL USA"
railmeets["Website Link"] = "https://morfracing.net/wpmocha/"

# Format Event Date explicitly as YYYY-MM-DD strings
railmeets["Event Date"] = (
    pd.to_datetime(morf["Date"], errors="coerce")
      .dt.strftime("%Y-%m-%d")
)

# Convert 'Start Time' like '6:30 PM' -> '18:30'
railmeets["Event Time"] = (
    pd.to_datetime(morf["Start Time"], errors="coerce")
      .dt.strftime("%H:%M")
)

railmeets

,Event Date,Event Time,Timezone,Fleet,Sponsoring Club,Series,Event,Race Type,Location,Website Link
0,2026-05-17,11:00,America/Chicago,MORF,Midwest Open Racing Fleet,MORF Performance,Performance Series - Race 1 (Olympic),Buoy,Chicago IL USA,https://morfracing.net/wpmocha/
1,2026-05-23,11:00,America/Chicago,MORF,Midwest Open Racing Fleet,MORF Performance,Performance Series - Race 2 (Btrap) Weathermar...,Buoy,Chicago IL USA,https://morfracing.net/wpmocha/
2,2026-05-23,NaN,America/Chicago,MORF,Midwest Open Racing Fleet,MORF Performance,Performance Series - Race 3 (W-L) Weathermark ...,Buoy,Chicago IL USA,https://morfracing.net/wpmocha/
3,2026-05-30,11:00,America/Chicago,MORF,Midwest Open Racing Fleet,MORF Performance,Performance Series - Race 4 (Trapezoid),Buoy,Chicago IL USA,https://morfracing.net/wpmocha/
4,2026-05-31,10:00,America/Chicago,MORF,Midwest Open Racing Fleet,MORF Long Distance,Long Distance Series - Race 1 (Zimmer),Distance (nearshore),Chicago IL USA,https://morfracing.net/wpmocha/
5,2026-05-31,10:00,America/Chicago,MORF,Midwest Open Racing Fleet,MORF Open Long Distance Race,Open Long Distance Race - (Zimmer),NaN,Chicago IL USA,https://morfracing.net/wpmocha/
6,2026-06-07,10:00,America/Chicago,MORF,Midwest Open Racing Fleet,MORF Long Distance,Long Distance Series - Race 2 (Skippers Club),Distance (nearshore),Chicago IL USA,https://morfracing.net/wpmocha/
7,2026-06-07,10:00,America/Chicago,MORF,Midwest Open Racing Fleet,MORF Open Long Distance Race,Open Long Distance Race - (Skippers Club),NaN,Chicago IL USA,https://morfracing.net/wpmocha/
8,2026-06-13,11:00,America/Chicago,MORF,Midwest Open Racing Fleet,MORF Performance,Performance Series - Race 5 (Btrap) Crowleys R...,Buoy,Chicago IL USA,https://morfracing.net/wpmocha/
9,2026-06-13,NaN,America/Chicago,MORF,Midwest Open Racing Fleet,MORF Performance,Performance Series - Race 6 (W-L) Crowleys Re...,Buoy,Chicago IL USA,https://morfracing.net/wpmocha/


In [15]:
# Check if 'Event' column exists and is populated
if "Event" not in railmeets.columns or railmeets["Event"].isnull().all():
    raise ValueError("The 'Event' column is missing or empty in the railmeets DataFrame.")

# Merge same-day races into single regatta entries
def merge_races(group):
    # Extract race numbers and suffixes
    races = group["Event"].dropna().tolist()
    race_numbers = []
    suffixes = set()
    
    for race in races:
        parts = race.split(" ", 1)
        if len(parts) > 1:
            race_numbers.append(parts[0])  # First part is the race number
            suffixes.add(parts[1])         # Second part is the suffix
        else:
            race_numbers.append(parts[0])  # Only race number exists
    
    # Combine race numbers into a range (e.g., "Race 2 & 3")
    if len(race_numbers) > 1:
        race_range = f"Race {race_numbers[0]} & {race_numbers[-1]}"
    else:
        race_range = f"Race {race_numbers[0]}"
    
    # Combine suffixes into a single string
    suffix = " ".join(sorted(suffixes))
    
    # Return the combined event name
    return f"{race_range} {suffix}".strip()

# Apply the merging logic
railmeets["Event"] = (
    railmeets.groupby("Event Date")["Event"]
    .transform(merge_races)
)

# Drop duplicate rows after merging
railmeets = railmeets.drop_duplicates(subset=["Event Date", "Event"])

KeyError: 'Event'

## Export to CSV

In [13]:
# Create folder if it doesn't exist
folder_name = "railmeet_upload"
os.makedirs(folder_name, exist_ok=True)

# Current year
year = datetime.now().year

# Build full file path with year in filename
file_path = os.path.join(folder_name, f"morf_railmeets_schedule_{year}.csv")

# Save CSV into that folder
railmeets.to_csv(file_path, index=False)

print(f"Saved file to: {file_path}")

Saved file to: railmeet_upload\morf_railmeets_schedule_2026.csv
